### Game Theory Assumptions - Kuhn Poker  ###

1) We will have a deck of only 3 cards - Jack, Queen King
2) We will only have 2 players 
3) Each Player has a set amount of chips of their choosing 
4) Jack < Queen < King
### Definitions ###
check - skips your turn and if both players check, the person with the higher card wins 
bet - place money or more money into the pot 
pot - place where you place the money (like a temorary bank)
### Starting the game ###
Intitial: Each player will enter a chip with equal amounts of money and be dealt a card each (2 cards dealth, 1 card hidden)
Betting round 1:
1) Player 1 acts first either choosing to check or bet 1 chip.
2) If player 1 checks then player 2 can check or bet.
3) If player 2 bets after player 1, P1 can fold or call.
4) If player 1 bets, player 2 can fold or call.
5) If at anypoint player 1 then player 2 checks, a showdown happens and it is revealed who has the higher card.
6) If at any point wither player folds, the other player gets whats in the pot (wins).
7) We repeat this for multiple rounds.



In [6]:
"""
Kuhn Poker payoff matrix generator.

This automates exactly what we did by hand:
  1. For a FIXED pair of strategies, trace all 6 card deals and average
     the payoffs (the "+1/6" calculation).
  2. Enumerate every pure strategy for each player.
  3. Run step 1 for every (P1 strategy, P2 strategy) pair to build the
     full payoff matrix A, where A[i][j] = expected payoff to P1.

By hand, you should still: derive the game tree, identify info sets,
and trace 1-2 deals manually to sanity-check this code against your
own numbers before trusting the full matrix.
"""

from itertools import product
import numpy as np

CARDS = ["J", "Q", "K"]
RANK = {"J": 0, "Q": 1, "K": 2}

# All 6 equally likely deals: every ordered pair of distinct cards
DEALS = [(p1c, p2c) for p1c in CARDS for p2c in CARDS if p1c != p2c]
DEAL_PROB = 1 / len(DEALS)  # = 1/6


def trace_payoff(p1c, p2c, p1_strategy, p2_strategy):
    """
    Walk one specific card deal under one fixed pair of strategies.
    Returns P1's net chip payoff for this single deal.

    p1_strategy = {
        "act1": {card: "bet"/"check"},          # P1's opening action
        "act2": {card: "fold"/"call"},          # P1's response if P1 checked and P2 bets
    }
    p2_strategy = {
        "vs_check": {card: "check"/"bet"},      # P2's action if P1 checked
        "vs_bet":   {card: "fold"/"call"},      # P2's action if P1 bet
    }
    """
    pot = 2          # both antes already in
    p1_contrib = 1
    p2_contrib = 1

    a1 = p1_strategy["act1"][p1c]

    if a1 == "bet":
        pot += 1
        p1_contrib += 1
        a2 = p2_strategy["vs_bet"][p2c]
        if a2 == "fold":
            return pot - p1_contrib          # P1 takes the pot
        else:  # call
            pot += 1
            p2_contrib += 1
            p1_wins = RANK[p1c] > RANK[p2c]
            return (pot - p1_contrib) if p1_wins else -p1_contrib

    else:  # a1 == "check"
        a2 = p2_strategy["vs_check"][p2c]
        if a2 == "check":
            p1_wins = RANK[p1c] > RANK[p2c]
            return (pot - p1_contrib) if p1_wins else -p1_contrib
        else:  # P2 bets
            pot += 1
            p2_contrib += 1
            a1b = p1_strategy["act2"][p1c]
            if a1b == "fold":
                return -p1_contrib
            else:  # call
                pot += 1
                p1_contrib += 1
                p1_wins = RANK[p1c] > RANK[p2c]
                return (pot - p1_contrib) if p1_wins else -p1_contrib


def expected_payoff(p1_strategy, p2_strategy):
    """Average trace_payoff over all 6 deals -- this is one matrix cell."""
    total = sum(
        trace_payoff(p1c, p2c, p1_strategy, p2_strategy)
        for (p1c, p2c) in DEALS
    )
    return total * DEAL_PROB


def all_p1_strategies():
    """Enumerate every pure strategy for P1: 2^3 (act1) x 2^3 (act2) = 64 raw combos."""
    act1_options = list(product(["bet", "check"], repeat=3))
    act2_options = list(product(["fold", "call"], repeat=3))
    strategies = []
    for a1 in act1_options:
        for a2 in act2_options:
            strategies.append({
                "act1": dict(zip(CARDS, a1)),
                "act2": dict(zip(CARDS, a2)),
            })
    return strategies


def all_p2_strategies():
    """Enumerate every pure strategy for P2: 2^3 (vs_check) x 2^3 (vs_bet) = 64 combos."""
    vs_check_options = list(product(["check", "bet"], repeat=3))
    vs_bet_options = list(product(["fold", "call"], repeat=3))
    strategies = []
    for vc in vs_check_options:
        for vb in vs_bet_options:
            strategies.append({
                "vs_check": dict(zip(CARDS, vc)),
                "vs_bet": dict(zip(CARDS, vb)),
            })
    return strategies


def strategy_label(strategy, player):
    """Short human-readable label for a strategy, for printing/debugging."""
    if player == 1:
        parts = [f"{c}:{strategy['act1'][c][0]}/{strategy['act2'][c][0]}" for c in CARDS]
    else:
        parts = [f"{c}:{strategy['vs_check'][c][0]}/{strategy['vs_bet'][c][0]}" for c in CARDS]
    return ",".join(parts)


def build_payoff_matrix(p1_strategies, p2_strategies):
    """Build the full m x n payoff matrix as a list of lists."""
    matrix = []
    for s1 in p1_strategies:
        row = [expected_payoff(s1, s2) for s2 in p2_strategies]
        matrix.append(row)
    return matrix


def build_payoff_matrix_np(p1_strategies, p2_strategies):
    """
    Same as build_payoff_matrix, but returns a numpy array directly.
    This is the form you'll feed into scipy.optimize.linprog later.
    """
    m, n = len(p1_strategies), len(p2_strategies)
    matrix = np.zeros((m, n))
    for i, s1 in enumerate(p1_strategies):
        for j, s2 in enumerate(p2_strategies):
            matrix[i, j] = expected_payoff(s1, s2)
    return matrix


def display_matrix(matrix, p1_strategies, p2_strategies, max_rows=10, max_cols=10):
    """
    Pretty-print a numpy payoff matrix with row/column strategy labels,
    aligned into a readable table. Truncates to max_rows x max_cols by
    default since the full 64x64 matrix is too wide to read comfortably.
    """
    m, n = matrix.shape
    show_rows = min(m, max_rows)
    show_cols = min(n, max_cols)

    row_labels = [strategy_label(s, 1) for s in p1_strategies[:show_rows]]
    col_labels = [strategy_label(s, 2) for s in p2_strategies[:show_cols]]

    row_label_width = max(len(lbl) for lbl in row_labels) + 2
    col_width = max(max(len(lbl) for lbl in col_labels), 8) + 2

    # Header row
    header = " " * row_label_width + "".join(f"{lbl:>{col_width}}" for lbl in col_labels)
    print(header)
    print("-" * len(header))

    # Body rows
    for i in range(show_rows):
        row_str = f"{row_labels[i]:<{row_label_width}}"
        for j in range(show_cols):
            row_str += f"{matrix[i, j]:>+{col_width}.3f}"
        print(row_str)

    if m > show_rows or n > show_cols:
        print(f"\n(showing {show_rows}x{show_cols} of full {m}x{n} matrix)")


if __name__ == "__main__":
    # Sanity check: reproduce the exact hand-computed example (+1/6)
    p1_example = {
        "act1": {"J": "check", "Q": "check", "K": "bet"},
        "act2": {"J": "fold", "Q": "fold", "K": "call"},
    }
    p2_example = {
        "vs_check": {"J": "check", "Q": "check", "K": "bet"},
        "vs_bet": {"J": "fold", "Q": "call", "K": "call"},
    }
    result = expected_payoff(p1_example, p2_example)
    print(f"Sanity check (should match hand calc ~0.1667): {result:.4f}")

    # Full enumeration -- this is the part that would take forever by hand
    p1_strats = all_p1_strategies()
    p2_strats = all_p2_strategies()
    print(f"\nTotal P1 strategies: {len(p1_strats)}")
    print(f"Total P2 strategies: {len(p2_strats)}")
    print(f"Full matrix size: {len(p1_strats)} x {len(p2_strats)} = {len(p1_strats)*len(p2_strats)} cells")

    matrix = build_payoff_matrix_np(p1_strats, p2_strats)
    print(f"\nMatrix type: {type(matrix)}, shape: {matrix.shape}, dtype: {matrix.dtype}")

    print("\nPretty-printed corner of the payoff matrix (P1's expected payoff):")
    display_matrix(matrix, p1_strats, p2_strats, max_rows=6, max_cols=6)

Sanity check (should match hand calc ~0.1667): 0.1667

Total P1 strategies: 64
Total P2 strategies: 64
Full matrix size: 64 x 64 = 4096 cells

Matrix type: <class 'numpy.ndarray'>, shape: (64, 64), dtype: float64

Pretty-printed corner of the payoff matrix (P1's expected payoff):
                     J:c/f,Q:c/f,K:c/f  J:c/f,Q:c/f,K:c/c  J:c/f,Q:c/c,K:c/f  J:c/f,Q:c/c,K:c/c  J:c/c,Q:c/f,K:c/f  J:c/c,Q:c/f,K:c/c
-------------------------------------------------------------------------------------------------------------------------------------
J:b/f,Q:b/f,K:b/f               +1.000             +0.000             +0.667             -0.333             +1.333             +0.333
J:b/f,Q:b/f,K:b/c               +1.000             +0.000             +0.667             -0.333             +1.333             +0.333
J:b/f,Q:b/c,K:b/f               +1.000             +0.000             +0.667             -0.333             +1.333             +0.333
J:b/f,Q:b/c,K:b/c               +1.000           

In [1]:
#Comparision with AI (Claude - Sonnet 4.6)
"""
Kuhn Poker payoff matrix generator.

This automates exactly what we did by hand:
  1. For a FIXED pair of strategies, trace all 6 card deals and average
     the payoffs (the "+1/6" calculation).
  2. Enumerate every pure strategy for each player.
  3. Run step 1 for every (P1 strategy, P2 strategy) pair to build the
     full payoff matrix A, where A[i][j] = expected payoff to P1.

By hand, you should still: derive the game tree, identify info sets,
and trace 1-2 deals manually to sanity-check this code against your
own numbers before trusting the full matrix.
"""

from itertools import product
import numpy as np

CARDS = ["J", "Q", "K"]
RANK = {"J": 0, "Q": 1, "K": 2}

# All 6 equally likely deals: every ordered pair of distinct cards
DEALS = [(p1c, p2c) for p1c in CARDS for p2c in CARDS if p1c != p2c]
DEAL_PROB = 1 / len(DEALS)  # = 1/6


def trace_payoff(p1c, p2c, p1_strategy, p2_strategy):
    """
    Walk one specific card deal under one fixed pair of strategies.
    Returns P1's net chip payoff for this single deal.

    p1_strategy = {
        "act1": {card: "bet"/"check"},          # P1's opening action
        "act2": {card: "fold"/"call"},          # P1's response if P1 checked and P2 bets
    }
    p2_strategy = {
        "vs_check": {card: "check"/"bet"},      # P2's action if P1 checked
        "vs_bet":   {card: "fold"/"call"},      # P2's action if P1 bet
    }
    """
    pot = 2          # both antes already in
    p1_contrib = 1
    p2_contrib = 1

    a1 = p1_strategy["act1"][p1c]

    if a1 == "bet":
        pot += 1
        p1_contrib += 1
        a2 = p2_strategy["vs_bet"][p2c]
        if a2 == "fold":
            return pot - p1_contrib          # P1 takes the pot
        else:  # call
            pot += 1
            p2_contrib += 1
            p1_wins = RANK[p1c] > RANK[p2c]
            return (pot - p1_contrib) if p1_wins else -p1_contrib

    else:  # a1 == "check"
        a2 = p2_strategy["vs_check"][p2c]
        if a2 == "check":
            p1_wins = RANK[p1c] > RANK[p2c]
            return (pot - p1_contrib) if p1_wins else -p1_contrib
        else:  # P2 bets
            pot += 1
            p2_contrib += 1
            a1b = p1_strategy["act2"][p1c]
            if a1b == "fold":
                return -p1_contrib
            else:  # call
                pot += 1
                p1_contrib += 1
                p1_wins = RANK[p1c] > RANK[p2c]
                return (pot - p1_contrib) if p1_wins else -p1_contrib


def expected_payoff(p1_strategy, p2_strategy):
    """Average trace_payoff over all 6 deals -- this is one matrix cell."""
    total = sum(
        trace_payoff(p1c, p2c, p1_strategy, p2_strategy)
        for (p1c, p2c) in DEALS
    )
    return total * DEAL_PROB


def mixed_expected_payoff(p1_mixed, p2_mixed):
    """
    Same idea as expected_payoff(), but for MIXED (probabilistic) strategies
    instead of pure ones. Needed to validate against the published Kuhn
    poker Nash equilibrium, which is mixed (e.g. "bet with J with
    probability alpha"), not a fixed action.

    p1_mixed = {
        "bet_prob":  {card: P(P1 bets with this card)},
        "call_prob": {card: P(P1 calls, given P1 checked and P2 bet)},
    }
    p2_mixed = {
        "bet_prob_vs_check": {card: P(P2 bets, given P1 checked)},
        "call_prob_vs_bet":  {card: P(P2 calls, given P1 bet)},
    }

    Works by computing the (fixed) payoff at each of the 5 distinct leaf
    types directly, then weighting them by the probability of reaching
    each leaf -- rather than branching on a single sampled action.
    """
    total = 0.0
    for p1c, p2c in DEALS:
        p1_wins = RANK[p1c] > RANK[p2c]

        b1 = p1_mixed["bet_prob"][p1c]            # P(P1 bets)
        f1 = p1_mixed["call_prob"][p1c]            # P(P1 calls | checked, faced bet)
        b2 = p2_mixed["bet_prob_vs_check"][p2c]     # P(P2 bets | P1 checked)
        c2 = p2_mixed["call_prob_vs_bet"][p2c]      # P(P2 calls | P1 bet)

        # Leaf payoffs (same chip accounting as trace_payoff)
        payoff_bet_fold = 1                                    # pot=3, p1_contrib=2
        payoff_bet_call = 2 if p1_wins else -2                  # pot=4, p1_contrib=2
        payoff_check_check = 1 if p1_wins else -1                # pot=2, p1_contrib=1
        payoff_check_bet_fold = -1                                # only ante lost
        payoff_check_bet_call = 2 if p1_wins else -2              # pot=4, p1_contrib=2

        branch_bet = c2 * payoff_bet_call + (1 - c2) * payoff_bet_fold
        branch_check = (1 - b2) * payoff_check_check + b2 * (
            f1 * payoff_check_bet_call + (1 - f1) * payoff_check_bet_fold
        )

        deal_ev = b1 * branch_bet + (1 - b1) * branch_check
        total += deal_ev

    return total * DEAL_PROB


def kuhn_equilibrium(alpha):
    """
    The published closed-form Kuhn poker Nash equilibrium, parameterised by
    alpha in [0, 1/3] -- P1's probability of bluff-betting with a Jack.
    Returns (p1_mixed, p2_mixed) ready for mixed_expected_payoff().
    """
    p1_mixed = {
        "bet_prob": {"J": alpha, "Q": 0.0, "K": 3 * alpha},
        "call_prob": {"J": 0.0, "Q": 1 / 3, "K": 1.0},
    }
    p2_mixed = {
        "bet_prob_vs_check": {"J": 1 / 3, "Q": 0.0, "K": 1.0},
        "call_prob_vs_bet": {"J": 0.0, "Q": 1 / 3, "K": 1.0},
    }
    return p1_mixed, p2_mixed



def all_p1_strategies():
    """Enumerate every pure strategy for P1: 2^3 (act1) x 2^3 (act2) = 64 raw combos."""
    act1_options = list(product(["bet", "check"], repeat=3))
    act2_options = list(product(["fold", "call"], repeat=3))
    strategies = []
    for a1 in act1_options:
        for a2 in act2_options:
            strategies.append({
                "act1": dict(zip(CARDS, a1)),
                "act2": dict(zip(CARDS, a2)),
            })
    return strategies


def all_p2_strategies():
    """Enumerate every pure strategy for P2: 2^3 (vs_check) x 2^3 (vs_bet) = 64 combos."""
    vs_check_options = list(product(["check", "bet"], repeat=3))
    vs_bet_options = list(product(["fold", "call"], repeat=3))
    strategies = []
    for vc in vs_check_options:
        for vb in vs_bet_options:
            strategies.append({
                "vs_check": dict(zip(CARDS, vc)),
                "vs_bet": dict(zip(CARDS, vb)),
            })
    return strategies


def strategy_label(strategy, player):
    """Short human-readable label for a strategy, for printing/debugging."""
    if player == 1:
        parts = [f"{c}:{strategy['act1'][c][0]}/{strategy['act2'][c][0]}" for c in CARDS]
    else:
        parts = [f"{c}:{strategy['vs_check'][c][0]}/{strategy['vs_bet'][c][0]}" for c in CARDS]
    return ",".join(parts)


def build_payoff_matrix(p1_strategies, p2_strategies):
    """Build the full m x n payoff matrix as a list of lists."""
    matrix = []
    for s1 in p1_strategies:
        row = [expected_payoff(s1, s2) for s2 in p2_strategies]
        matrix.append(row)
    return matrix


def build_payoff_matrix_np(p1_strategies, p2_strategies):
    """
    Same as build_payoff_matrix, but returns a numpy array directly.
    This is the form you'll feed into scipy.optimize.linprog later.
    """
    m, n = len(p1_strategies), len(p2_strategies)
    matrix = np.zeros((m, n))
    for i, s1 in enumerate(p1_strategies):
        for j, s2 in enumerate(p2_strategies):
            matrix[i, j] = expected_payoff(s1, s2)
    return matrix


def display_matrix(matrix, p1_strategies, p2_strategies, max_rows=10, max_cols=10):
    """
    Pretty-print a numpy payoff matrix with row/column strategy labels,
    aligned into a readable table. Truncates to max_rows x max_cols by
    default since the full 64x64 matrix is too wide to read comfortably.
    """
    m, n = matrix.shape
    show_rows = min(m, max_rows)
    show_cols = min(n, max_cols)

    row_labels = [strategy_label(s, 1) for s in p1_strategies[:show_rows]]
    col_labels = [strategy_label(s, 2) for s in p2_strategies[:show_cols]]

    row_label_width = max(len(lbl) for lbl in row_labels) + 2
    col_width = max(max(len(lbl) for lbl in col_labels), 8) + 2

    # Header row
    header = " " * row_label_width + "".join(f"{lbl:>{col_width}}" for lbl in col_labels)
    print(header)
    print("-" * len(header))

    # Body rows
    for i in range(show_rows):
        row_str = f"{row_labels[i]:<{row_label_width}}"
        for j in range(show_cols):
            row_str += f"{matrix[i, j]:>+{col_width}.3f}"
        print(row_str)

    if m > show_rows or n > show_cols:
        print(f"\n(showing {show_rows}x{show_cols} of full {m}x{n} matrix)")


if __name__ == "__main__":
    # Sanity check: reproduce the exact hand-computed example (+1/6)
    p1_example = {
        "act1": {"J": "check", "Q": "check", "K": "bet"},
        "act2": {"J": "fold", "Q": "fold", "K": "call"},
    }
    p2_example = {
        "vs_check": {"J": "check", "Q": "check", "K": "bet"},
        "vs_bet": {"J": "fold", "Q": "call", "K": "call"},
    }
    result = expected_payoff(p1_example, p2_example)
    print(f"Sanity check (should match hand calc ~0.1667): {result:.4f}")

    # Validate against the PUBLISHED Kuhn poker Nash equilibrium (mixed strategy)
    alpha = 1 / 6  # canonical midpoint of the valid range [0, 1/3]
    p1_eq, p2_eq = kuhn_equilibrium(alpha)
    game_value = mixed_expected_payoff(p1_eq, p2_eq)
    print(f"\nPublished equilibrium game value (alpha={alpha:.4f}): {game_value:.4f}")
    print(f"Expected (known result): -1/18 = {-1/18:.4f}")
    print(f"Match: {abs(game_value - (-1/18)) < 1e-9}")


    # Full enumeration -- this is the part that would take forever by hand
    p1_strats = all_p1_strategies()
    p2_strats = all_p2_strategies()
    print(f"\nTotal P1 strategies: {len(p1_strats)}")
    print(f"Total P2 strategies: {len(p2_strats)}")
    print(f"Full matrix size: {len(p1_strats)} x {len(p2_strats)} = {len(p1_strats)*len(p2_strats)} cells")

    matrix = build_payoff_matrix_np(p1_strats, p2_strats)
    print(f"\nMatrix type: {type(matrix)}, shape: {matrix.shape}, dtype: {matrix.dtype}")

    print("\nPretty-printed corner of the payoff matrix (P1's expected payoff):")
    display_matrix(matrix, p1_strats, p2_strats, max_rows=6, max_cols=6)

Sanity check (should match hand calc ~0.1667): 0.1667

Published equilibrium game value (alpha=0.1667): -0.0556
Expected (known result): -1/18 = -0.0556
Match: True

Total P1 strategies: 64
Total P2 strategies: 64
Full matrix size: 64 x 64 = 4096 cells

Matrix type: <class 'numpy.ndarray'>, shape: (64, 64), dtype: float64

Pretty-printed corner of the payoff matrix (P1's expected payoff):
                     J:c/f,Q:c/f,K:c/f  J:c/f,Q:c/f,K:c/c  J:c/f,Q:c/c,K:c/f  J:c/f,Q:c/c,K:c/c  J:c/c,Q:c/f,K:c/f  J:c/c,Q:c/f,K:c/c
-------------------------------------------------------------------------------------------------------------------------------------
J:b/f,Q:b/f,K:b/f               +1.000             +0.000             +0.667             -0.333             +1.333             +0.333
J:b/f,Q:b/f,K:b/c               +1.000             +0.000             +0.667             -0.333             +1.333             +0.333
J:b/f,Q:b/c,K:b/f               +1.000             +0.000             +0

### Comparision Insights ###
Matrix had slight differences, AI added a validation check to show the expected value of -1/18 which helps as it suggests being P2 has a slight probabilistic advantage.

In [3]:
matrix

array([[ 1.        ,  0.        ,  0.66666667, ...,  0.33333333,
         1.        ,  0.        ],
       [ 1.        ,  0.        ,  0.66666667, ...,  0.33333333,
         1.        ,  0.        ],
       [ 1.        ,  0.        ,  0.66666667, ...,  0.33333333,
         1.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -0.33333333,
        -0.33333333, -0.33333333],
       [ 0.        ,  0.        ,  0.        , ..., -1.        ,
        -1.        , -1.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(64, 64))

In [4]:
"""
Kuhn Poker payoff matrix generator.

This automates exactly what we did by hand:
  1. For a FIXED pair of strategies, trace all 6 card deals and average
     the payoffs (the "+1/6" calculation).
  2. Enumerate every pure strategy for each player.
  3. Run step 1 for every (P1 strategy, P2 strategy) pair to build the
     full payoff matrix A, where A[i][j] = expected payoff to P1.

By hand, you should still: derive the game tree, identify info sets,
and trace 1-2 deals manually to sanity-check this code against your
own numbers before trusting the full matrix.
"""

from itertools import product
import numpy as np
from scipy.optimize import linprog

CARDS = ["J", "Q", "K"]
RANK = {"J": 0, "Q": 1, "K": 2}

# All 6 equally likely deals: every ordered pair of distinct cards
DEALS = [(p1c, p2c) for p1c in CARDS for p2c in CARDS if p1c != p2c]
DEAL_PROB = 1 / len(DEALS)  # = 1/6


def trace_payoff(p1c, p2c, p1_strategy, p2_strategy):
    """
    Walk one specific card deal under one fixed pair of strategies.
    Returns P1's net chip payoff for this single deal.

    p1_strategy = {
        "act1": {card: "bet"/"check"},          # P1's opening action
        "act2": {card: "fold"/"call"},          # P1's response if P1 checked and P2 bets
    }
    p2_strategy = {
        "vs_check": {card: "check"/"bet"},      # P2's action if P1 checked
        "vs_bet":   {card: "fold"/"call"},      # P2's action if P1 bet
    }
    """
    pot = 2          # both antes already in
    p1_contrib = 1
    p2_contrib = 1

    a1 = p1_strategy["act1"][p1c]

    if a1 == "bet":
        pot += 1
        p1_contrib += 1
        a2 = p2_strategy["vs_bet"][p2c]
        if a2 == "fold":
            return pot - p1_contrib          # P1 takes the pot
        else:  # call
            pot += 1
            p2_contrib += 1
            p1_wins = RANK[p1c] > RANK[p2c]
            return (pot - p1_contrib) if p1_wins else -p1_contrib

    else:  # a1 == "check"
        a2 = p2_strategy["vs_check"][p2c]
        if a2 == "check":
            p1_wins = RANK[p1c] > RANK[p2c]
            return (pot - p1_contrib) if p1_wins else -p1_contrib
        else:  # P2 bets
            pot += 1
            p2_contrib += 1
            a1b = p1_strategy["act2"][p1c]
            if a1b == "fold":
                return -p1_contrib
            else:  # call
                pot += 1
                p1_contrib += 1
                p1_wins = RANK[p1c] > RANK[p2c]
                return (pot - p1_contrib) if p1_wins else -p1_contrib



In [5]:

def expected_payoff(p1_strategy, p2_strategy):
    """Average trace_payoff over all 6 deals -- this is one matrix cell."""
    total = sum(
        trace_payoff(p1c, p2c, p1_strategy, p2_strategy)
        for (p1c, p2c) in DEALS
    )
    return total * DEAL_PROB


def mixed_expected_payoff(p1_mixed, p2_mixed):
    """
    Same idea as expected_payoff(), but for MIXED (probabilistic) strategies
    instead of pure ones. Needed to validate against the published Kuhn
    poker Nash equilibrium, which is mixed (e.g. "bet with J with
    probability alpha"), not a fixed action.

    p1_mixed = {
        "bet_prob":  {card: P(P1 bets with this card)},
        "call_prob": {card: P(P1 calls, given P1 checked and P2 bet)},
    }
    p2_mixed = {
        "bet_prob_vs_check": {card: P(P2 bets, given P1 checked)},
        "call_prob_vs_bet":  {card: P(P2 calls, given P1 bet)},
    }

    Works by computing the (fixed) payoff at each of the 5 distinct leaf
    types directly, then weighting them by the probability of reaching
    each leaf -- rather than branching on a single sampled action.
    """
    total = 0.0
    for p1c, p2c in DEALS:
        p1_wins = RANK[p1c] > RANK[p2c]

        b1 = p1_mixed["bet_prob"][p1c]            # P(P1 bets)
        f1 = p1_mixed["call_prob"][p1c]            # P(P1 calls | checked, faced bet)
        b2 = p2_mixed["bet_prob_vs_check"][p2c]     # P(P2 bets | P1 checked)
        c2 = p2_mixed["call_prob_vs_bet"][p2c]      # P(P2 calls | P1 bet)

        # Leaf payoffs (same chip accounting as trace_payoff)
        payoff_bet_fold = 1                                    # pot=3, p1_contrib=2
        payoff_bet_call = 2 if p1_wins else -2                  # pot=4, p1_contrib=2
        payoff_check_check = 1 if p1_wins else -1                # pot=2, p1_contrib=1
        payoff_check_bet_fold = -1                                # only ante lost
        payoff_check_bet_call = 2 if p1_wins else -2              # pot=4, p1_contrib=2

        branch_bet = c2 * payoff_bet_call + (1 - c2) * payoff_bet_fold
        branch_check = (1 - b2) * payoff_check_check + b2 * (
            f1 * payoff_check_bet_call + (1 - f1) * payoff_check_bet_fold
        )

        deal_ev = b1 * branch_bet + (1 - b1) * branch_check
        total += deal_ev

    return total * DEAL_PROB



In [6]:

def kuhn_equilibrium(alpha):
    """
    The published closed-form Kuhn poker Nash equilibrium, parameterised by
    alpha in [0, 1/3] -- P1's probability of bluff-betting with a Jack.
    Returns (p1_mixed, p2_mixed) ready for mixed_expected_payoff().
    """
    p1_mixed = {
        "bet_prob": {"J": alpha, "Q": 0.0, "K": 3 * alpha},
        "call_prob": {"J": 0.0, "Q": 1 / 3, "K": 1.0},
    }
    p2_mixed = {
        "bet_prob_vs_check": {"J": 1 / 3, "Q": 0.0, "K": 1.0},
        "call_prob_vs_bet": {"J": 0.0, "Q": 1 / 3, "K": 1.0},
    }
    return p1_mixed, p2_mixed




In [ ]:
#added an Primal LP solver, AI used for structure, documentaiion and in-line comments 
def solve_primal_lp(A):
    """
    Solve P1's maximin LP using scipy.optimize.linprog.

    Primal (as derived):
        max_{x,v}  v
        s.t.       sum_i x_i * A[i][j] >= v   for every column j
                    sum_i x_i = 1
                    x_i >= 0

    linprog only minimises and only accepts <= constraints, so this gets
    translated mechanically:
        - maximise v  ->  minimise -v
        - sum_i x_i*A[i][j] >= v  ->  -sum_i x_i*A[i][j] + v <= 0

    Returns:
        x: P1's optimal mixed strategy (probability per row/strategy)
        v: the guaranteed game value to P1
        y: P2's optimal mixed strategy, recovered from the dual values
        w: the dual objective (should equal v, by strong duality)
        res: the raw linprog result object
    """
    m, n = A.shape

    # Variables are [x_1, ..., x_m, v] -- length m+1
    c = np.zeros(m + 1)
    c[-1] = -1  # minimise -v == maximise v

    # One <= constraint per column j: -A[:,j]^T x + v <= 0
    A_ub = np.hstack([-A.T, np.ones((n, 1))])
    b_ub = np.zeros(n)

    # Equality: sum_i x_i = 1
    A_eq = np.zeros((1, m + 1))
    A_eq[0, :m] = 1
    b_eq = [1]

    # x_i >= 0 ; v is unrestricted in sign
    bounds = [(0, None)] * m + [(None, None)]

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                   bounds=bounds, method="highs")

    x = res.x[:m]
    v = res.x[-1]

    # Dual values for the column constraints ARE P2's optimal strategy y
    # (this is the "free" dual output strong duality promised us).
    # highs reports marginals as <= 0 for <= constraints; flip sign to get
    # a proper non-negative probability vector.
    y = -res.ineqlin.marginals
    w = v  # strong duality guarantees these are equal; verified numerically below

    return x, v, y, w, res




In [8]:

def all_p1_strategies():
    """Enumerate every pure strategy for P1: 2^3 (act1) x 2^3 (act2) = 64 raw combos."""
    act1_options = list(product(["bet", "check"], repeat=3))
    act2_options = list(product(["fold", "call"], repeat=3))
    strategies = []
    for a1 in act1_options:
        for a2 in act2_options:
            strategies.append({
                "act1": dict(zip(CARDS, a1)),
                "act2": dict(zip(CARDS, a2)),
            })
    return strategies


def all_p2_strategies():
    """Enumerate every pure strategy for P2: 2^3 (vs_check) x 2^3 (vs_bet) = 64 combos."""
    vs_check_options = list(product(["check", "bet"], repeat=3))
    vs_bet_options = list(product(["fold", "call"], repeat=3))
    strategies = []
    for vc in vs_check_options:
        for vb in vs_bet_options:
            strategies.append({
                "vs_check": dict(zip(CARDS, vc)),
                "vs_bet": dict(zip(CARDS, vb)),
            })
    return strategies


def strategy_label(strategy, player):
    """Short human-readable label for a strategy, for printing/debugging."""
    if player == 1:
        parts = [f"{c}:{strategy['act1'][c][0]}/{strategy['act2'][c][0]}" for c in CARDS]
    else:
        parts = [f"{c}:{strategy['vs_check'][c][0]}/{strategy['vs_bet'][c][0]}" for c in CARDS]
    return ",".join(parts)



In [9]:

def build_payoff_matrix(p1_strategies, p2_strategies):
    """Build the full m x n payoff matrix as a list of lists."""
    matrix = []
    for s1 in p1_strategies:
        row = [expected_payoff(s1, s2) for s2 in p2_strategies]
        matrix.append(row)
    return matrix


def build_payoff_matrix_np(p1_strategies, p2_strategies):
    """
    Same as build_payoff_matrix, but returns a numpy array directly.
    This is the form you'll feed into scipy.optimize.linprog later.
    """
    m, n = len(p1_strategies), len(p2_strategies)
    matrix = np.zeros((m, n))
    for i, s1 in enumerate(p1_strategies):
        for j, s2 in enumerate(p2_strategies):
            matrix[i, j] = expected_payoff(s1, s2)
    return matrix


def display_matrix(matrix, p1_strategies, p2_strategies, max_rows=10, max_cols=10):
    """
    Pretty-print a numpy payoff matrix with row/column strategy labels,
    aligned into a readable table. Truncates to max_rows x max_cols by
    default since the full 64x64 matrix is too wide to read comfortably.
    """
    m, n = matrix.shape
    show_rows = min(m, max_rows)
    show_cols = min(n, max_cols)

    row_labels = [strategy_label(s, 1) for s in p1_strategies[:show_rows]]
    col_labels = [strategy_label(s, 2) for s in p2_strategies[:show_cols]]

    row_label_width = max(len(lbl) for lbl in row_labels) + 2
    col_width = max(max(len(lbl) for lbl in col_labels), 8) + 2

    # Header row
    header = " " * row_label_width + "".join(f"{lbl:>{col_width}}" for lbl in col_labels)
    print(header)
    print("-" * len(header))

    # Body rows
    for i in range(show_rows):
        row_str = f"{row_labels[i]:<{row_label_width}}"
        for j in range(show_cols):
            row_str += f"{matrix[i, j]:>+{col_width}.3f}"
        print(row_str)

    if m > show_rows or n > show_cols:
        print(f"\n(showing {show_rows}x{show_cols} of full {m}x{n} matrix)")


if __name__ == "__main__":
    # Sanity check: reproduce the exact hand-computed example (+1/6)
    p1_example = {
        "act1": {"J": "check", "Q": "check", "K": "bet"},
        "act2": {"J": "fold", "Q": "fold", "K": "call"},
    }
    p2_example = {
        "vs_check": {"J": "check", "Q": "check", "K": "bet"},
        "vs_bet": {"J": "fold", "Q": "call", "K": "call"},
    }
    result = expected_payoff(p1_example, p2_example)
    print(f"Sanity check (should match hand calc ~0.1667): {result:.4f}")

    # Validate against the PUBLISHED Kuhn poker Nash equilibrium (mixed strategy)
    alpha = 1 / 6  # canonical midpoint of the valid range [0, 1/3]
    p1_eq, p2_eq = kuhn_equilibrium(alpha)
    game_value = mixed_expected_payoff(p1_eq, p2_eq)
    print(f"\nPublished equilibrium game value (alpha={alpha:.4f}): {game_value:.4f}")
    print(f"Expected (known result): -1/18 = {-1/18:.4f}")
    print(f"Match: {abs(game_value - (-1/18)) < 1e-9}")


    # Full enumeration -- this is the part that would take forever by hand
    p1_strats = all_p1_strategies()
    p2_strats = all_p2_strategies()
    print(f"\nTotal P1 strategies: {len(p1_strats)}")
    print(f"Total P2 strategies: {len(p2_strats)}")
    print(f"Full matrix size: {len(p1_strats)} x {len(p2_strats)} = {len(p1_strats)*len(p2_strats)} cells")

    matrix = build_payoff_matrix_np(p1_strats, p2_strats)
    print(f"\nMatrix type: {type(matrix)}, shape: {matrix.shape}, dtype: {matrix.dtype}")

    print("\nPretty-printed corner of the payoff matrix (P1's expected payoff):")
    display_matrix(matrix, p1_strats, p2_strats, max_rows=6, max_cols=6)

    # --- Week 2: solve the primal LP and recover the dual ---
    x, v, y, w, res = solve_primal_lp(matrix)
    print(f"\nLP status: {res.message}")
    print(f"Guaranteed game value to P1 (v): {v:.4f}")
    print(f"Dual game value to P2 (w):       {w:.4f}")
    print(f"Compare to known value -1/18:    {-1/18:.4f}")
    print(f"x sums to 1: {x.sum():.6f}   y sums to 1: {y.sum():.6f}")

    print("\nP1's solved strategy -- top 5 strategies by probability:")
    top_p1 = np.argsort(-x)[:5]
    for idx in top_p1:
        print(f"  x={x[idx]:.4f}  {strategy_label(p1_strats[idx], 1)}")

    print("\nP2's solved strategy -- top 5 strategies by probability:")
    top_p2 = np.argsort(-y)[:5]
    for idx in top_p2:
        print(f"  y={y[idx]:.4f}  {strategy_label(p2_strats[idx], 2)}")

Sanity check (should match hand calc ~0.1667): 0.1667

Published equilibrium game value (alpha=0.1667): -0.0556
Expected (known result): -1/18 = -0.0556
Match: True

Total P1 strategies: 64
Total P2 strategies: 64
Full matrix size: 64 x 64 = 4096 cells

Matrix type: <class 'numpy.ndarray'>, shape: (64, 64), dtype: float64

Pretty-printed corner of the payoff matrix (P1's expected payoff):
                     J:c/f,Q:c/f,K:c/f  J:c/f,Q:c/f,K:c/c  J:c/f,Q:c/c,K:c/f  J:c/f,Q:c/c,K:c/c  J:c/c,Q:c/f,K:c/f  J:c/c,Q:c/f,K:c/c
-------------------------------------------------------------------------------------------------------------------------------------
J:b/f,Q:b/f,K:b/f               +1.000             +0.000             +0.667             -0.333             +1.333             +0.333
J:b/f,Q:b/f,K:b/c               +1.000             +0.000             +0.667             -0.333             +1.333             +0.333
J:b/f,Q:b/c,K:b/f               +1.000             +0.000             +0